# 🎙️ Wav2Lip Studio - Khớp Khẩu Hình Video (Tự Động Lưu Vào Google Drive)
> **Hướng dẫn 1-Click (Không cần tải lại lần sau):**
> 1. Bấm nút **'Sao chép vào Drive'** ở thanh trên cùng để lưu vĩnh viễn notebook này vào Google Drive của bạn.
> 2. Bấm **Runtime (Thời gian chạy)** -> **Thay đổi loại phần cứng** -> Chọn **T4 GPU**.
> 3. Bấm nút ▶️ ở ô lệnh bên dưới -> Bấm **'Kết nối với Google Drive'** khi được hỏi.
> 4. Lần đầu sẽ lưu model vào Drive, **từ lần thứ 2 trở đi sẽ load tức thì từ Drive trong 3 giây!**

In [ ]:
#@title 🚀 Khởi chạy Wav2Lip WebUI 1-Click (Tự Động Caching Google Drive)
import os
import shutil
from google.colab import drive

# 1. Gắn kết Google Drive thông minh
print("🔗 Đang kiểm tra kết nối Google Drive...")
has_drive = False
if os.path.exists('/content/drive/MyDrive'):
    print("🎉 Google Drive đã được gắn kết từ trước! Bỏ qua xác thực...")
    has_drive = True
else:
    try:
        drive.mount('/content/drive')
        has_drive = True
    except Exception as e:
        print(f"⚠️ Không thể gắn kết Drive tự động ({e}). Sẽ chạy trên bộ nhớ tạm của Colab.")

if has_drive:
    drive_cache_dir = "/content/drive/MyDrive/AI_Colab_Cache/Wav2Lip"
    os.makedirs(drive_cache_dir, exist_ok=True)
else:
    drive_cache_dir = "/content/cache/Wav2Lip"
    os.makedirs(drive_cache_dir, exist_ok=True)

# 2. Clone mã nguồn Wav2Lip (chỉ lấy commit mới nhất --depth 1 để siêu tốc)
%cd /content
if not os.path.exists("/content/Wav2Lip"):
    print("⚡ Đang nạp mã nguồn Wav2Lip...")
    !git clone --depth 1 https://huggingface.co/camenduru/Wav2Lip /content/Wav2Lip

%cd /content/Wav2Lip
os.makedirs("/content/Wav2Lip/results", exist_ok=True)

# 3. Kiểm tra Caching Model
if os.path.exists(f"{drive_cache_dir}/checkpoints/wav2lip_gan.pth"):
    print("🎉 ĐÃ TÌM THẤY MODEL TRONG GOOGLE DRIVE! Nạp trực tiếp trong 3 giây...")
    !cp -r "{drive_cache_dir}/checkpoints" /content/Wav2Lip/
else:
    print("⏳ Đang lưu bản sao model để lần sau không phải tải lại...")
    os.makedirs(f"{drive_cache_dir}/checkpoints", exist_ok=True)
    !cp -r /content/Wav2Lip/checkpoints "{drive_cache_dir}/"

# 4. Cài đặt thư viện môi trường (Ép buộc nâng cấp Gradio 4+)
print("📦 Đang nâng cấp thư viện lên bản mới nhất...")
!pip install -q -U "gradio>=4.0.0" yt_dlp ffmpeg-python librosa

# 5. Vá lỗi librosa hiện đại trong audio.py
!sed -i "s/librosa.filters.mel(hp.sample_rate, hp.n_fft/librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft/g" /content/Wav2Lip/audio.py
!sed -i "s/librosa.core.load/librosa.load/g" /content/Wav2Lip/audio.py

# 6. Tải file app.py chuẩn từ kho và khởi chạy WebUI
print("🚀 Đang khởi động Wav2Lip WebUI...")
!curl -s -L -o /content/Wav2Lip/app.py https://raw.githubusercontent.com/nviethiep55-glitch/my-ai-studio-colab/main/wav2lip_app.py
!python /content/Wav2Lip/app.py
